In [21]:
import os
import logging, timeit
#from btEngine2.DataLoader import DataLoader
from btEngine2.MarketData import MarketData
from btEngine2.TradingRule import TradingRule

import polars as pl
import numpy as np  


import platform
import pandas as pd

pd.options.display.float_format = lambda x: f'{x:,.0f}' if abs(x) >= 1000 else (f'{x:.2f}' if abs(x) < 10 else f'{x:.1f}')
# Detect operating system
if platform.system() == "Windows":
    ticker_csv_path = r'G:\Projects\BackTesting1.0\Data\Inputs\TickerList-Futs.csv'
    save_directory = r"G:\Projects\BackTesting1.0\Data\Bloomberg\Futures"
    helper_directory = r'G:\Projects\BackTesting1.0\Data\Bloomberg\HelperFiles'
    bt_folder = r'BackTests\simple_mr'
    av_folder = r'G:\Projects\BackTesting1.0\Data\Inputs\AssetSizing-Futs.csv'
else:  # Assume macOS for other cases
    ticker_csv_path = r'Data/Inputs/TickerList-Futs.csv'
    save_directory = r"Data/Bloomberg/Futures"
    helper_directory = r'Data/Bloomberg/HelperFiles'
    bt_folder = r'BackTests/simple_mr'
    av_folder = r'Data/Inputs/AssetSizing-Futs.csv'



# Define paths to auxiliary data for MarketData
tick_values_path = os.path.join(helper_directory, 'fut_val_pt.parquet')
fx_rates_path = os.path.join(helper_directory, 'fxHist.parquet')

# Initialize the MarketData
market_data = MarketData(
    base_directory=save_directory,
    tick_values_path=tick_values_path,
    fx_rates_path=fx_rates_path,
    instrument_type="Futures",
    n_threads=8,  # Number of threads for parallel data loading
    log_level=logging.ERROR  # Set to DEBUG for more detailed logs
)

tick = 'US1 Comdty'
# Access data for a specific ticker
try:
    test_df = market_data.get_ticker_data(tick)
    print(test_df)
except ValueError as e:
    print(e)

# Access all preprocessed data
all_data = market_data.get_data()
print(f"Total tickers loaded: {len(all_data)}")

# Access FX rates
fx_rates = market_data.get_fx_rates()
# Access tick values
tick_values = market_data.get_tick_values()
# Access asset classes
asset_classes = market_data.get_asset_classes()

#market_data = market_data.date_filter(start_date='01012010')

shape: (11_317, 14)
┌────────────┬───────────┬───────────┬───────────┬───┬─────────┬─────────┬────────────┬────────────┐
│ Date       ┆ Open      ┆ High      ┆ Low       ┆ … ┆ BadOHLC ┆ FX_Rate ┆ Tick_Value ┆ Tick_Value │
│ ---        ┆ ---       ┆ ---       ┆ ---       ┆   ┆ ---     ┆ ---     ┆ _Base      ┆ _USD       │
│ date       ┆ f64       ┆ f64       ┆ f64       ┆   ┆ bool    ┆ f64     ┆ ---        ┆ ---        │
│            ┆           ┆           ┆           ┆   ┆         ┆         ┆ f64        ┆ f64        │
╞════════════╪═══════════╪═══════════╪═══════════╪═══╪═════════╪═════════╪════════════╪════════════╡
│ 1980-01-02 ┆ -41.875   ┆ -41.875   ┆ -43.09375 ┆ … ┆ false   ┆ 1.0     ┆ 1000.0     ┆ 1000.0     │
│ 1980-01-03 ┆ -43.46875 ┆ -43.34375 ┆ -43.9375  ┆ … ┆ false   ┆ 1.0     ┆ 1000.0     ┆ 1000.0     │
│ 1980-01-04 ┆ -43.625   ┆ -43.15625 ┆ -43.9375  ┆ … ┆ false   ┆ 1.0     ┆ 1000.0     ┆ 1000.0     │
│ 1980-01-07 ┆ -44.5     ┆ -43.625   ┆ -44.5     ┆ … ┆ false   ┆ 1.0   

In [22]:
import pandas as pd
import numpy as np
import polars as pl

def rsi_trading_rule(
    df: pl.DataFrame,
    N: int,
    thresholds: tuple,
    mode: str,
    h: int = 1,
    sig_delay: int = 0,
    trade_direction: str = 'both'
) -> pl.DataFrame:
    """
    Implements a trading rule based on RSI.

    Parameters:
    - df: Polars DataFrame with 'Date', 'Open', 'High', 'Low', 'Close' columns.
    - N: RSI lookback period.
    - thresholds: Tuple (low_t, high_t), RSI thresholds.
    - mode: 'mr' for mean reversion, 'bo' for breakout.
    - h: Holding period in days, default is 1.
    - sig_delay: Signal delay in days, default is 0.
    - trade_direction: 'both', 'long', or 'short', default is 'both'.

    Returns:
    - Polars DataFrame with added 'Signal', 'TradeEntry', 'TradeExit', 'InTrade' columns.
    """
    # Convert Polars DataFrame to Pandas DataFrame
    df = df.to_pandas()

    # Ensure 'Date' column is datetime and set as index
    if 'Date' in df.columns:
        df['Date'] = pd.to_datetime(df['Date'])
        df.set_index('Date', inplace=True)
    else:
        df.index = pd.to_datetime(df.index)

    # Calculate RSI
    delta = df['Close'].diff()
    gain = np.where(delta > 0, delta, 0)
    loss = np.where(delta < 0, -delta, 0)

    # Use exponential moving average
    roll_up = pd.Series(gain, index=df.index).ewm(span=N, adjust=False).mean()
    roll_down = pd.Series(loss, index=df.index).ewm(span=N, adjust=False).mean()

    RS = roll_up / roll_down
    RSI = 100 - (100 / (1 + RS))
    df['RSI'] = RSI

    # Generate signals
    low_t, high_t = thresholds
    df['Signal'] = 0

    if mode == 'mr':
        # Mean Reversion
        df.loc[df['RSI'] < low_t, 'Signal'] = 1  # Buy
        df.loc[df['RSI'] > high_t, 'Signal'] = -1  # Sell
    elif mode == 'bo':
        # Breakout
        df.loc[df['RSI'] < low_t, 'Signal'] = -1  # Sell
        df.loc[df['RSI'] > high_t, 'Signal'] = 1  # Buy
    else:
        raise ValueError("Invalid mode. Must be 'mr' or 'bo'.")

    # Apply signal delay if any
    if sig_delay != 0:
        df['Signal'] = df['Signal'].shift(sig_delay)

    # Apply trade direction
    if trade_direction == 'long':
        df.loc[df['Signal'] == -1, 'Signal'] = 0
    elif trade_direction == 'short':
        df.loc[df['Signal'] == 1, 'Signal'] = 0
    elif trade_direction != 'both':
        raise ValueError("Invalid trade_direction. Must be 'long', 'short', or 'both'.")

    # Initialize TradeEntry, TradeExit, InTrade columns
    df['TradeEntry'] = np.nan
    df['TradeExit'] = np.nan
    df['InTrade'] = 0

    # Implement trading logic
    in_trade = False
    trade_start_idx = None
    trade_end_idx = None

    df = df.reset_index()

    for i in range(len(df)):
        signal = df.loc[i, 'Signal']

        if not in_trade and signal != 0:
            # Enter trade at next open
            if i + 1 < len(df):
                trade_entry_idx = i + 1
                df.loc[trade_entry_idx, 'TradeEntry'] = df.loc[trade_entry_idx, 'Open']
                in_trade = True
                trade_start_idx = trade_entry_idx

                # Calculate trade_end_idx
                trade_end_idx = trade_entry_idx + h - 1
                if trade_end_idx >= len(df):
                    trade_end_idx = len(df) - 1

                df.loc[trade_end_idx, 'TradeExit'] = df.loc[trade_end_idx, 'Close']
                df.loc[trade_end_idx, 'Signal'] = -signal
                # Set InTrade
                df.loc[trade_start_idx:trade_end_idx, 'InTrade'] = signal

        elif in_trade and i == trade_end_idx:
            in_trade = False
            trade_start_idx = None
            trade_end_idx = None
    # Convert back to Polars DataFrame
    return pl.from_pandas(df)

In [63]:
test_rule = rsi_trading_rule(
    df=test_df,
    N=7,
    thresholds=(10, 90),
    mode='mr',
    h=2,
    sig_delay=0,
    trade_direction='long'
).to_pandas()

test_rule.set_index('Date', inplace=True)

In [64]:
test_rule.to_clipboard()

0         NaN
1         NaN
2       -0.19
3         NaN
4         NaN
         ... 
11312     NaN
11313     NaN
11314     NaN
11315     NaN
11316     NaN
Name: StratPnL, Length: 11317, dtype: float64